In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data import SliceConditionDataset
from inference import sample_conditioned_latent_volume, voxel_to_latent_index
from models import CustomVAE, DDPM, SliceConditionedTimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
PATCH_SIZE = 64
TIMESTEPS = 3
AXIS = "z"
AXIS_ID = 0
SLICE_INDEX = 12
SEED = 0


In [ ]:
dataset = SliceConditionDataset(DATA_DIR, patch_size=PATCH_SIZE, axis=AXIS, slice_index=SLICE_INDEX, seed=SEED)
batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False)))
condition = batch["condition"]
condition.shape, batch["axis"], batch["slice_index"]


In [ ]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = SliceConditionedTimeUNet(latent_ch=4, base_ch=16, time_dim=16, max_slices=64)
unet.eval()
ddpm = DDPM(timesteps=TIMESTEPS)


In [ ]:
with torch.no_grad():
    condition_z, _ = vae.encode(condition * 2 - 1)
    volume_z = sample_conditioned_latent_volume(
        unet,
        ddpm,
        condition_z.squeeze(0),
        axis=AXIS_ID,
        slice_index=SLICE_INDEX,
        volume_shape=(4, 16, 16, 16),
    )
    latent_index = voxel_to_latent_index(SLICE_INDEX)
    fixed_error = (volume_z[:, latent_index, :, :] - condition_z.squeeze(0)).abs().max()

volume_z.shape, condition_z.shape, latent_index, float(fixed_error)
